#Statement of Intent
My goal is to build an end-to-end ETL pipeline in PySpark to process player performance data and generate predictive features. Trained and compared multiple ML models (Random Forest, Gradient Boosting, Logistic Regression) to predict match outcomes. Later model versions will integrate MLflow for model tracking and cross-validation to evaluate performance. Designed for deployment to support real-time match predictions or possibly used to bet for the moneyline, though the latter is not the desired purpose.

#Data Sourcing
The data is sourced from match statistics provided by Jeff Sackmann.

The last 4 years worth of matches from Jeff's GitHub repo have been stored in s3 on a free AWS account. I am starting the feature engineering by initiating the spark session and pulling the raw files from the s3 bucket.

In [0]:
from pyspark.sql import (
    SparkSession,
    types,
    functions as F,
)

from pyspark.sql.window import Window

I convert the files to delta for faster pulling, version control, and updating records if needed. I only need to run that cell once.

In [0]:
#Convert to delta
df = spark.read.csv("s3://data-storage-for-projects/Tennis Analytics Project/v1/raw/"
                    , header=True, inferSchema=True)
df.write.format("delta").mode("overwrite").save("s3://data-storage-for-projects/Tennis Analytics Project/v1/delta/")

In [0]:
#Bring data into notebook
df = spark.read.format("delta").load("s3://data-storage-for-projects/Tennis Analytics Project/v1/delta/")
display(df)

#Baseline Models aka V1
For my baseline models, I am using tournament conditions such as surface, tournament level, best of, and round of the match. I will also include rankings of each player, but I will need to have two rows per match so that each player is player 1. The data in its current form has winners all in one column and could train the model incorrectly. As for the types of models, I plan to create and compare three--logistic regression, random forest, and gradient boosting.

In [0]:
#Columns for player a and player b rank, age, and hand. It also says if a won. Two are used to prevent data leakage.
d1 = df.select('surface', 'tourney_level', 'round',
        'best_of', 'winner_hand', 'loser_hand',
        F.col('winner_id').alias('id_a'), F.col('loser_id').alias('id_b'), 
        F.col('winner_rank').alias('id_a_rank'), F.col('loser_rank').alias('id_b_rank'),
        F.col('winner_age').alias('id_a_age'), F.col('loser_age').alias('id_b_age')
        ).withColumn('rank_diff', F.col('id_a_rank') - F.col('id_b_rank')) \
        .withColumn('best_of', F.when(F.col('best_of')==3, 0).otherwise(1)) \
        .withColumn('id_a_hand', F.when(F.col('winner_hand')=='R', 0).otherwise(1)) \
        .withColumn('id_b_hand', F.when(F.col('loser_hand')=='R', 0).otherwise(1))  \
        .withColumn("round_encoded",
        F.when(df.round == "RR", 0)
        .when(df.round == "R128", 1)
        .when(df.round == "R64", 2)
        .when(df.round == "R32", 3)
        .when(df.round == "R16", 4)
        .when(df.round == "QF", 5)
        .when(df.round == "SF", 6)
        .when(df.round == "BR", 6.5)
        .when(df.round == "F", 7)
        .otherwise(None)
        ) \
        .withColumn('id_a_won', F.lit(1))


d2 = df.select('surface', 'tourney_level', 'round',
        'best_of', 'winner_hand', 'loser_hand',
        F.col('loser_id').alias('id_a'), F.col('winner_id').alias('id_b'), 
        F.col('loser_rank').alias('id_a_rank'), F.col('winner_rank').alias('id_b_rank'),
        F.col('loser_age').alias('id_a_age'), F.col('winner_age').alias('id_b_age')
        ).withColumn('rank_diff', F.col('id_a_rank') - F.col('id_b_rank')) \
        .withColumn('best_of', F.when(F.col('best_of')==3, 0).otherwise(1)) \
        .withColumn('id_a_hand', F.when(F.col('loser_hand')=='R', 0).otherwise(1)) \
        .withColumn('id_b_hand', F.when(F.col('winner_hand')=='R', 0).otherwise(1)) \
        .withColumn("round_encoded",
        F.when(F.col('round') == "RR", 0)
        .when(F.col('round') == "R128", 1)
        .when(F.col('round') == "R64", 2)
        .when(F.col('round') == "R32", 3)
        .when(F.col('round') == "R16", 4)
        .when(F.col('round') == "QF", 5)
        .when(F.col('round') == "SF", 6)
        .when(F.col('round') == "BR", 6.5)
        .when(F.col('round') == "F", 7)
        .otherwise(None)
        ) \
        .withColumn('id_a_won', F.lit(0))

d = d1.union(d2).drop('winner_hand', 'loser_hand', 'round') \
        .filter(F.col('surface').isNotNull() & F.col('rank_diff').isNotNull()
                & F.col('id_a_age').isNotNull() & F.col('id_b_age').isNotNull()) 
display(d)

In [0]:
#Create ML Pipeline starting with training data, string indexer and one hot encoder for logisitic regression
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

#Traininng and testing data
train_df, test_df = d.randomSplit([.8, .2], seed=42)

#Indexing and Encoding
indexer_surface = StringIndexer(inputCol='surface', outputCol='surface_index')
encoder_surface = OneHotEncoder(inputCol='surface_index', outputCol='surface_vec')

indexer_tourney = StringIndexer(inputCol='tourney_level', outputCol='tourney_index')
encoder_tourney = OneHotEncoder(inputCol='tourney_index', outputCol='tourney_vec')

#X variables
feature_cols = ['surface_vec', 'tourney_vec', 'round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand']

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

In [0]:
#This function will be used later to evaluate each model through a variety of metrics
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics

def evaluate_model(predictions, model_name=''):
    # ==========================================
    # 1. AUC Metrics (BinaryClassificationEvaluator)
    # ==========================================
    auc_roc_evaluator = BinaryClassificationEvaluator(labelCol='id_a_won', metricName='areaUnderROC')
    auc_pr_evaluator = BinaryClassificationEvaluator(labelCol='id_a_won', metricName='areaUnderPR')

    auc_roc = auc_roc_evaluator.evaluate(predictions)
    auc_pr = auc_pr_evaluator.evaluate(predictions)

    # ==========================================
    # 2. Accuracy, Precision, Recall, F1 (MulticlassClassificationEvaluator)
    # ==========================================
    accuracy_evaluator = MulticlassClassificationEvaluator(labelCol='id_a_won', predictionCol='prediction', metricName='accuracy')
    precision_evaluator = MulticlassClassificationEvaluator(labelCol='id_a_won', predictionCol='prediction', metricName='weightedPrecision')
    recall_evaluator = MulticlassClassificationEvaluator(labelCol='id_a_won', predictionCol='prediction', metricName='weightedRecall')
    f1_evaluator = MulticlassClassificationEvaluator(labelCol='id_a_won', predictionCol='prediction', metricName='f1')

    accuracy = accuracy_evaluator.evaluate(predictions)
    precision = precision_evaluator.evaluate(predictions)
    recall = recall_evaluator.evaluate(predictions)
    f1 = f1_evaluator.evaluate(predictions)

    # ==========================================
    # 3. Confusion Matrix (manual calculation)
    # ==========================================
    predictions.groupBy('id_a_won', 'prediction').count().show()

    # Or create it properly:
    tp = predictions.filter((F.col('id_a_won') == 1) & (F.col('prediction') == 1)).count()
    tn = predictions.filter((F.col('id_a_won') == 0) & (F.col('prediction') == 0)).count()
    fp = predictions.filter((F.col('id_a_won') == 0) & (F.col('prediction') == 1)).count()
    fn = predictions.filter((F.col('id_a_won') == 1) & (F.col('prediction') == 0)).count()

    print("=" * 60)
    print(f"{model_name.upper()} - EVALUATION METRICS")
    print("=" * 60)
    print(f"AUC-ROC:              {auc_roc:.4f}")
    print(f"AUC-PR:               {auc_pr:.4f}")
    print(f"Accuracy:             {accuracy:.4f}")
    print(f"Precision (Weighted): {precision:.4f}")
    print(f"Recall (Weighted):    {recall:.4f}")
    print(f"F1 Score:             {f1:.4f}")
    print("=" * 60)
    print("CONFUSION MATRIX")
    print("=" * 60)
    print(f"True Positives:       {tp:,}")
    print(f"True Negatives:       {tn:,}")
    print(f"False Positives:      {fp:,}")
    print(f"False Negatives:      {fn:,}")
    print("=" * 60)

    return {
        'auc_roc': auc_roc,
        'auc_pr': auc_pr,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': {'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}
    }

In [0]:
#Logistic Regression Model
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol='features', labelCol='id_a_won')

pipeline_lr = Pipeline(stages = [
    indexer_surface, encoder_surface,
    indexer_tourney, encoder_tourney,
    assembler,
    lr
])

#Create Model and save to s3
lr_model = pipeline_lr.fit(train_df)
lr_model.save("s3://data-storage-for-projects/Tennis Analytics Project/v1/v1-models/logistic-regressions/1.0/")

#Generate Predictions and Evaluate Accuracy
lr_predict = lr_model.transform(test_df)
lr_eval = evaluate_model(lr_predict, 'Logistic Regression')

#Remove from Databricks Memory to create more models
del lr_model, lr_predict

In [0]:
#Random Forest
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol='features', labelCol='id_a_won', numTrees=50, maxDepth=10)

feature_cols = ['surface_index', 'tourney_index', 'round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand']

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

pipeline_rf = Pipeline(stages = [
    indexer_surface,
    indexer_tourney,
    assembler,
    rf
])

#Create Model and save to s3
rf_model = pipeline_rf.fit(train_df)
rf_model.save("s3://data-storage-for-projects/Tennis Analytics Project/v1/v1-models/random-forests/1.1/")

#Generate Predictions and Evaluate Accuracy
rf_predict = rf_model.transform(test_df)
rf_eval = evaluate_model(rf_predict, 'Random Forest')

#Remove from Databricks Memory to create more models
del rf_model, rf_predict

In [0]:
#Gradient Boosting
from pyspark.ml.classification import GBTClassifier

gb = GBTClassifier(labelCol="id_a_won", featuresCol="features", maxDepth=6, maxIter=20)

feature_cols = ['surface_index', 'tourney_index', 'round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand']

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

pipeline_gb = Pipeline(stages = [
    indexer_surface,
    indexer_tourney,
    assembler,
    gb
])

#Create Model and save to s3
gb_model = pipeline_gb.fit(train_df)
gb_model.save("s3://data-storage-for-projects/Tennis Analytics Project/v1/v1-models/gradient-boostings/1.0/")

#Generate Predictions and Evaluate Accuracy
gb_predict = gb_model.transform(test_df)
gb_eval = evaluate_model(gb_predict, 'Gradient Boosting')

#Remove from Databricks Memory to create more models
del gb_model, gb_predict

The evaluations for the three models are shown below.
| **Metric**             | **Logistic Regression** | **Random Forest** | **Gradient Boosting** |
|--------------------------|--------------------------|--------------------|------------------------|
| **AUC-ROC**              | 0.6698                   | 0.6953             | 0.6967                 |
| **AUC-PR**               | 0.6824                   | 0.7056             | 0.7066                 |
| **Accuracy**             | 0.6178                   | 0.6283             | 0.6338                 |
| **Precision (Weighted)** | 0.6213                   | 0.6299             | 0.6358                 |
| **Recall (Weighted)**    | 0.6178                   | 0.6283             | 0.6338                 |
| **F1 Score**             | 0.6175                   | 0.6285             | 0.6339                 |
| **True Positives**       | 1,386                    | 1,467              | 1,466                  |
| **True Negatives**       | 1,436                    | 1,403              | 1,429                  |
| **False Positives**      | 743                      | 776                | 750                    |
| **False Negatives**      | 1,003                    | 922                | 923                    |

Gradient Boosting appears to be the best v1 model, evidenced by the higher scores in AUC-ROC and F1. Random Forest is not far behind, and the results are all higher than 50%, suggesting they are better than chance. However, there is room for improvement.

#V2
For this round of models, there are several factors that I would like to add to increase the evaluation scores. The most obvious is to add more data. Instead of four years let's see what six or eight years provides. A time series cross validation will be implemented instead of one random 80/20 split. I would also like to generate running totals of stats before the match is counted. This would include win percentage, 1st serve percentage, and break percentage for the last ten matches. Using the statistics for the match being predicted would result in data leakage, so I cannot use those until I factor them in for the next match.

In [0]:
#Convert to delta
df = spark.read.csv("s3://data-storage-for-projects/Tennis Analytics Project/v2/raw/"
                    , header=True, inferSchema=True)
df.write.format("delta").mode("overwrite").save("s3://data-storage-for-projects/Tennis Analytics Project/v2/delta/")

In [0]:
#Bring data into notebook
df = spark.read.format("delta").load("s3://data-storage-for-projects/Tennis Analytics Project/v2/delta/")
display(df)

In [0]:
df = df.withColumn('tourney_date', F.to_date(F.col('tourney_date').cast('string'), 'yyyyMMdd')) \
    .withColumn("id", F.monotonically_increasing_id()) \
    .withColumn("round_encoded",
        F.when(F.col('round') == "RR", 0)
        .when(F.col('round') == "R128", 1)
        .when(F.col('round') == "R64", 2)
        .when(F.col('round') == "R32", 3)
        .when(F.col('round') == "R16", 4)
        .when(F.col('round') == "QF", 5)
        .when(F.col('round') == "SF", 6)
        .when(F.col('round') == "BR", 6.5)
        .when(F.col('round') == "F", 7)
        .otherwise(None))

df1 = df.select(F.col('winner_id').alias('player_id'), 'id', 'tourney_date', 'round_encoded', 'minutes', F.col('w_df').alias('df'), 
                F.col('w_bpFaced').alias('bp_faced'), F.col('l_bpFaced').alias('bp_created'),
                'w_1stIn', 'w_svpt', 'w_1stWon', 'w_2ndWon', 'w_bpSaved',
                'l_bpSaved', 'l_svpt', 'l_1stWon', 'l_2ndWon', 'l_SvGms'
 ) \
    .withColumn('first_pct', F.try_divide(F.col('w_1stIn'), F.col('w_svpt'))) \
    .withColumn('first_win_pct', F.try_divide(F.col('w_1stWon'), F.col('w_1stIn'))) \
    .withColumn('second_win_pct', F.try_divide(F.col('w_2ndWon'), F.col('w_svpt') - F.col('w_1stIn'))) \
    .withColumn('bp_saved_pct', F.try_divide(F.col('w_bpSaved'), F.col('bp_faced'))) \
    .withColumn('bp_convert_pct', F.try_divide(F.col('bp_created') - F.col('l_bpSaved'), F.col('bp_created'))) \
    .withColumn('return_pt_win_pct', F.try_divide(F.col('l_svpt') - F.col('l_1stWon') - F.col('l_2ndWon'), F.col('l_svpt'))) \
    .withColumn('break_pct', F.try_divide(F.col('bp_created') - F.col('l_bpSaved'), F.col('l_SvGms'))) \
    .drop('w_1stIn', 'w_svpt', 'w_1stWon', 'w_2ndWon', 'w_bpSaved', 'l_bpSaved', 'l_svpt', 'l_1stWon', 'l_2ndWon', 'l_SvGms')

df2 = (df.select(F.col('loser_id').alias('player_id'), 'id', 'tourney_date', 'round_encoded', 'minutes', F.col('l_df').alias('df'), 
                F.col('l_bpFaced').alias('bp_faced'), F.col('w_bpFaced').alias('bp_created'),
                'l_1stIn', 'l_svpt', 'l_1stWon', 'l_2ndWon', 'l_bpSaved',
                'w_bpSaved', 'w_svpt', 'w_1stWon', 'w_2ndWon', 'w_SvGms'
 ) \
    .withColumn('first_pct', F.try_divide(F.col('l_1stIn'), F.col('l_svpt'))) \
    .withColumn('first_win_pct', F.try_divide(F.col('l_1stWon'), F.col('l_1stIn'))) \
    .withColumn('second_win_pct', F.try_divide(F.col('l_2ndWon'), F.col('l_svpt') - F.col('l_1stIn'))) \
    .withColumn('bp_saved_pct', F.try_divide(F.col('l_bpSaved'), F.col('bp_faced'))) \
    .withColumn('bp_convert_pct', F.try_divide(F.col('bp_created') - F.col('w_bpSaved'), F.col('bp_created'))) \
    .withColumn('return_pt_win_pct', F.try_divide(F.col('w_svpt') - F.col('w_1stWon') - F.col('w_2ndWon'), F.col('w_svpt'))) \
    .withColumn('break_pct', F.try_divide(F.col('bp_created') - F.col('w_bpSaved'), F.col('w_SvGms'))) \
    .drop('l_1stIn', 'l_svpt', 'l_1stWon', 'l_2ndWon', 'l_bpSaved', 'w_bpSaved', 'w_svpt', 'w_1stWon', 'w_2ndWon', 'w_SvGms'))

player = df1.union(df2)
display(player)

In [0]:
#Stats for last 5 matches
w = Window.partitionBy('player_id').orderBy('tourney_date', 'round_encoded').rowsBetween(-5, -1)

player = player.withColumn('first_pct_avg', F.avg('first_pct').over(w)) \
    .withColumn('first_win_pct_avg', F.avg('first_win_pct').over(w)) \
    .withColumn('second_win_pct_avg', F.avg('second_win_pct').over(w)) \
    .withColumn('bp_saved_pct_avg', F.avg('bp_saved_pct').over(w)) \
    .withColumn('bp_convert_pct_avg', F.avg('bp_convert_pct').over(w)) \
    .withColumn('return_pt_win_pct_avg', F.avg('return_pt_win_pct').over(w)) \
    .withColumn('break_pct_avg', F.avg('break_pct').over(w)) \
    .withColumn('df_avg', F.avg('df').over(w)) \
    .withColumn('bp_faced_avg', F.avg('bp_faced').over(w)) \
    .withColumn('bp_created_avg', F.avg('bp_created').over(w)) \
    .withColumn('minutes_sum', F.sum('minutes').over(w))

display(player)

In [0]:
d = df.join(
    player.alias('pa'),
    (df.winner_id == F.col('pa.player_id')) & (df.id == F.col('pa.id')),
    'left'
).join(
    player.alias('pb'),
    (df.loser_id == F.col('pb.player_id')) & (df.id == F.col('pb.id')),
    'left'
).select(
    # Keep df columns (not duplicated)
        df.id,
        df.round,
        df.surface,
        df.tourney_level,
        df.best_of,
        df.winner_id,
        df.loser_id,
        df.winner_rank,
        df.loser_rank,
        df.winner_age,
        df.loser_age,
        df.winner_hand,
        df.loser_hand,
    
    # Winner's rolling stats from 'pa'
    F.col('pa.first_pct_avg').alias('winner_first_pct_avg'),
    F.col('pa.first_win_pct_avg').alias('winner_first_win_pct_avg'),
    F.col('pa.second_win_pct_avg').alias('winner_second_win_pct_avg'),
    F.col('pa.bp_saved_pct_avg').alias('winner_bp_saved_pct_avg'),
    F.col('pa.bp_convert_pct_avg').alias('winner_bp_convert_pct_avg'),
    F.col('pa.return_pt_win_pct_avg').alias('winner_return_pt_win_pct_avg'),
    F.col('pa.break_pct_avg').alias('winner_break_pct_avg'),
    F.col('pa.df_avg').alias('winner_df_avg'),
    F.col('pa.minutes_sum').alias('winner_minutes_sum'),
    
    # Loser's rolling stats from 'pb'
    F.col('pb.first_pct_avg').alias('loser_first_pct_avg'),
    F.col('pb.first_win_pct_avg').alias('loser_first_win_pct_avg'),
    F.col('pb.second_win_pct_avg').alias('loser_second_win_pct_avg'),
    F.col('pb.bp_saved_pct_avg').alias('loser_bp_saved_pct_avg'),
    F.col('pb.bp_convert_pct_avg').alias('loser_bp_convert_pct_avg'),
    F.col('pb.return_pt_win_pct_avg').alias('loser_return_pt_win_pct_avg'),
    F.col('pb.break_pct_avg').alias('loser_break_pct_avg'),
    F.col('pb.df_avg').alias('loser_df_avg'),
    F.col('pb.minutes_sum').alias('loser_minutes_sum')
)\
.withColumn('random_flip', F.rand(seed=42))

d = d.select('*') \
.withColumn('id_a', F.when(F.col('random_flip')>= 0.5, F.col('winner_id')).otherwise(F.col('loser_id'))) \
.withColumn('id_b', F.when(F.col('random_flip')>= 0.5, F.col('loser_id')).otherwise(F.col('winner_id'))) \
.withColumn('id_a_rank', F.when(F.col('random_flip')>= 0.5, F.col('winner_rank')).otherwise(F.col('loser_rank'))) \
.withColumn('id_b_rank', F.when(F.col('random_flip')>= 0.5, F.col('loser_rank')).otherwise(F.col('winner_rank'))) \
.withColumn('id_a_age', F.when(F.col('random_flip')>= 0.5, F.col('winner_age')).otherwise(F.col('loser_age'))) \
.withColumn('id_b_age', F.when(F.col('random_flip')>= 0.5, F.col('loser_age')).otherwise(F.col('winner_age'))) \
.withColumn('rank_diff', F.col('id_a_rank') - F.col('id_b_rank')) \
.withColumn('best_of', F.when(F.col('best_of')==3, 0).otherwise(1)) \
.withColumn('id_a_hand', F.when(F.col('random_flip')>= 0.5, F.when(F.col('winner_hand')=='R', 0).otherwise(1)).otherwise(F.when(F.col('loser_hand')=='R', 0).otherwise(1))) \
.withColumn('id_b_hand', F.when(F.col('random_flip')>= 0.5, F.when(F.col('loser_hand')=='R', 0).otherwise(1)).otherwise(F.when(F.col('winner_hand')=='R', 0).otherwise(1)))  \
.withColumn('id_a_first_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_first_pct_avg')).otherwise(F.col('loser_first_pct_avg'))) \
.withColumn('id_b_first_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_first_pct_avg')).otherwise(F.col
('winner_first_pct_avg'))) \
.withColumn('id_a_first_win_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_first_win_pct_avg')).otherwise(F.col('loser_first_win_pct_avg'))) \
.withColumn('id_b_first_win_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_first_win_pct_avg')).otherwise(F.col
('winner_first_win_pct_avg'))) \
.withColumn('id_a_second_win_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_second_win_pct_avg')).otherwise(F.col('loser_second_win_pct_avg'))) \
.withColumn('id_b_second_win_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_second_win_pct_avg')).otherwise(F.col
('winner_second_win_pct_avg'))) \
.withColumn('id_a_bp_saved_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_bp_saved_pct_avg')).otherwise(F.col('loser_bp_saved_pct_avg'))) \
.withColumn('id_b_bp_saved_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_bp_saved_pct_avg')).otherwise(F.col
('winner_bp_saved_pct_avg'))) \
.withColumn('id_a_bp_convert_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_bp_convert_pct_avg')).otherwise(F.col('loser_bp_convert_pct_avg'))) \
.withColumn('id_b_bp_convert_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_bp_convert_pct_avg')).otherwise(F.col
('winner_bp_convert_pct_avg'))) \
.withColumn('id_a_return_pt_win_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_return_pt_win_pct_avg')).otherwise(F.col('loser_return_pt_win_pct_avg'))) \
.withColumn('id_b_return_pt_win_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_return_pt_win_pct_avg')).otherwise(F.col
('winner_return_pt_win_pct_avg'))) \
.withColumn('id_a_break_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_break_pct_avg')).otherwise(F.col('loser_break_pct_avg'))) \
.withColumn('id_b_break_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_break_pct_avg')).otherwise(F.col
('winner_break_pct_avg'))) \
.withColumn('id_a_df_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_df_avg')).otherwise(F.col('loser_df_avg'))) \
.withColumn('id_b_df_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_df_avg')).otherwise(F.col
('winner_df_avg'))) \
.withColumn('id_a_minutes_sum', F.when(F.col('random_flip')>= 0.5, F.col('winner_minutes_sum')).otherwise(F.col('loser_minutes_sum'))) \
.withColumn('id_b_minutes_sum', F.when(F.col('random_flip')>= 0.5, F.col('loser_minutes_sum')).otherwise(F.col('winner_minutes_sum'))) \
.withColumn('minutes_diff', F.col('id_a_minutes_sum') - F.col('id_b_minutes_sum')) \
.withColumn("round_encoded",
        F.when(F.col('round') == "RR", 0)
        .when(F.col('round') == "R128", 1)
        .when(F.col('round') == "R64", 2)
        .when(F.col('round') == "R32", 3)
        .when(F.col('round') == "R16", 4)
        .when(F.col('round') == "QF", 5)
        .when(F.col('round') == "SF", 6)
        .when(F.col('round') == "BR", 6.5)
        .when(F.col('round') == "F", 7)
        .otherwise(None)) \
.withColumn('id_a_won', F.when(F.col('random_flip')>= 0.5, 1).otherwise(0)) \
.drop('winner_hand', 'loser_hand', 'winner_age', 'loser_age', 'winner_rank', 'loser_rank', 'random_flip',
      'winner_first_pct_avg', 'loser_first_pct_avg', 'winner_second_pct_avg', 'loser_second_pct_avg',
      'winner_bp_saved_pct_avg', 'loser_bp_saved_pct_avg', 'winner_bp_convert_pct_avg', 'loser_bp_convert_pct_avg',
      'winner_return_pt_win_pct_avg', 'loser_return_pt_win_pct_avg', 'winner_break_pct_avg', 'loser_break_pct_avg',
      'winner_df_avg', 'loser_df_avg', 'winner_minutes_sum', 'loser_minutes_sum', 'winner_first_win_pct_avg',
      'loser_first_win_pct_avg', 'winner_second_win_pct_avg', 'loser_second_win_pct_avg', 'winner_id', 'loser_id',
      'id', 'round') \
.filter(F.col('surface').isNotNull() & F.col('rank_diff').isNotNull()
        & F.col('id_a_age').isNotNull() & F.col('id_b_age').isNotNull()) 

display(d)